In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
import json
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import math
import random
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold

from pathlib import Path
from tqdm import tqdm

In [6]:
# --- 1. CONFIGURATION ---
CONFIG = {
    # Data Paths
    'base_data_path': '/content/drive/MyDrive/Google_AI_Studio/ARCAGI2025/data',
    'input_directory': 'GridTransitionDataset/training_transformed_unique_ids',
    'output_directory': '/content/drive/MyDrive/Google_AI_Studio/ARCAGI2025/models',

    # Model Hyperparameters
    'vocab_size': 12,
    'max_seq_len': 1802,
    'd_model': 128,         # adjustable
    'nhead': 4,
    'num_layers': 4,        # adjustable
    'dim_feedforward': 512, # adjustable
    'dropout': 0.1,

    # Training Parameters
    'batch_size': 16, # adjustable
    'learning_rate': 0.0002,
    'num_epochs': 10,       # adjustable
    'n_splits': 5,
    'train_generator_every': 1,
    'label_smoothing': 0.1, # Added label smoothing parameter
    'gradient_clipping_value': 1.0, # Added gradient clipping parameter
    'gradient_accumulation_steps': 4, # Added gradient accumulation steps
    'checkpoint_interval': 1 # Save checkpoint every X epochs
}

In [7]:
# --- 2. DATASET CLASS ---
class ARCDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]

In [8]:
# --- 3. MODEL ARCHITECTURE ---
class Generator(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, dim_feedforward, dropout):
        super().__init__()
        # Embedding layer to convert discrete tokens to vectors
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        # Transformer Encoder to process the embedded sequence
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        # Output layer to convert back to token logits
        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, src):
        # src is expected to be a LongTensor of token indices
        src = self.token_embedding(src)
        output = self.transformer_encoder(src)
        logits = self.output_layer(output)
        return logits

class Discriminator(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, dim_feedforward, dropout):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        self.output_layer = nn.Linear(d_model, 1)

    def forward(self, src):
        src = self.token_embedding(src)
        output = self.transformer_encoder(src)
        output = torch.mean(output, dim=1)
        return self.output_layer(output)


In [9]:
# --- 4. WEIGHTS INITIALIZATION ---
def weights_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.Embedding):
        nn.init.xavier_uniform_(m.weight)

In [10]:
# --- 5. TRAINER CLASS ---
class GANTrainer:
    def __init__(self, config):
        self.config = config
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        os.makedirs(self.config['output_directory'], exist_ok=True)

        # Pre-filter and load all data into a list in memory
        print("--- Pre-loading and Caching All Data ---")
        all_file_paths = glob.iglob(
            os.path.join(self.config['base_data_path'], self.config['input_directory'], '**', '*.json'),
            recursive=True
        )
        self.all_sequences = self._load_all_sequences(list(all_file_paths))
        print(f"Total sequences loaded: {len(self.all_sequences)}")

        # Initialize final_generator to None
        self.final_generator = None


    def _load_all_sequences(self, file_paths):
        all_sequences = []
        for file_path in tqdm(file_paths, desc="Loading and Caching Data"):
            try:
                with open(file_path, 'r') as f:
                    task = json.load(f)
                    # Check if 'train' key exists and is not empty
                    if 'train' in task and task['train']:
                        for example in task['train']:
                            # Check if 'input' and 'output' keys exist and are not empty
                            if 'input' in example and example['input'] and 'output' in example and example['output']:
                                input_grid = np.array(example['input'])
                                output_grid = np.array(example['output'])

                                # Flatten and concatenate the grids
                                start_token, sep_token, pad_token = 10, 11, 0
                                input_flat = input_grid.flatten()
                                output_flat = output_grid.flatten()

                                sequence = np.concatenate([
                                    [start_token],
                                    input_flat,
                                    [sep_token],
                                    output_flat
                                ])

                                # Pad sequence to max length
                                if len(sequence) > CONFIG['max_seq_len']:
                                    # Truncate if sequence is longer than max_seq_len
                                    sequence = sequence[:CONFIG['max_seq_len']]

                                padded_sequence = np.pad(sequence, (0, CONFIG['max_seq_len'] - len(sequence)), 'constant', constant_values=pad_token)
                                all_sequences.append(torch.tensor(padded_sequence, dtype=torch.long))
                            else:
                                # print(f"Skipping example in {file_path} due to missing or empty 'input' or 'output'.") # Commented out to reduce output clutter
                                pass
                    else:
                        # print(f"Skipping file {file_path} due to missing or empty 'train' key.") # Commented out to reduce output clutter
                        pass
            except json.JSONDecodeError:
                print(f"Error decoding JSON from file: {file_path}")
                continue
            except Exception as e:
                print(f"Error processing file {file_path}: {e}")
                continue
        return all_sequences

    def run_cross_validation(self):
        k_fold = KFold(n_splits=self.config['n_splits'], shuffle=True, random_state=42)

        fold_discriminator_losses = []
        fold_generator_losses = []

        if len(self.all_sequences) == 0:
            print("No sequences loaded. Cannot perform cross-validation.")
            return

        for fold, (train_ids, val_ids) in enumerate(k_fold.split(self.all_sequences)):
            print(f"--- FOLD {fold+1}/{self.config['n_splits']} ---")

            generator = Generator(
                self.config['vocab_size'], self.config['d_model'], self.config['nhead'],
                self.config['num_layers'], self.config['dim_feedforward'], self.config['dropout']
            ).to(self.device)
            discriminator = Discriminator(
                self.config['vocab_size'], self.config['d_model'], self.config['nhead'],
                self.config['num_layers'], self.config['dim_feedforward'], self.config['dropout']
            ).to(self.device)

            optimizer_G = optim.AdamW(generator.parameters(), lr=self.config['learning_rate'])
            optimizer_D = optim.AdamW(discriminator.parameters(), lr=self.config['learning_rate'])
            criterion = nn.BCEWithItlogitsLoss()

            # Learning rate schedulers
            scheduler_G = CosineAnnealingLR(optimizer_G, T_max=self.config['num_epochs'])
            scheduler_D = CosineAnnealingLR(optimizer_D, T_max=self.config['num_epochs'])

            # Gradient scaler for mixed precision training
            scaler = GradScaler()

            # Check for existing checkpoint for this fold
            start_epoch = self.load_checkpoint(generator, discriminator, optimizer_G, optimizer_D, scheduler_G, scheduler_D, fold + 1)

            if start_epoch > 0:
                print(f"Resuming training from epoch {start_epoch + 1}")
            else:
                generator.apply(weights_init)
                discriminator.apply(weights_init)


            train_subsampler = torch.utils.data.SubsetRandomSampler(train_ids)
            val_subsampler = torch.utils.data.SubsetRandomSampler(val_ids)

            dataset = ARCDataset(self.all_sequences)

            train_loader = DataLoader(dataset, batch_size=self.config['batch_size'], sampler=train_subsampler)
            val_loader = DataLoader(dataset, batch_size=self.config['batch_size'], sampler=val_loader)


            avg_fold_disc_loss, avg_fold_gen_loss = self.train_gan(
                generator, discriminator, optimizer_G, optimizer_D, criterion, train_loader,
                scheduler_G, scheduler_D, scaler, fold + 1, start_epoch
            )

            fold_discriminator_losses.append(avg_fold_disc_loss)
            fold_generator_losses.append(avg_fold_gen_loss)

            self.save_models(generator, discriminator, fold + 1)

        print("\n--- Cross-Validation Results ---")
        avg_disc_loss_cv = sum(fold_discriminator_losses) / self.config['n_splits']
        avg_gen_loss_cv = sum(fold_generator_losses) / self.config['n_splits']
        print(f"Average Discriminator Loss across {self.config['n_splits']} folds: {avg_disc_loss_cv:.4f}")
        print(f"Average Generator Loss across {self.config['n_splits']} folds: {avg_gen_loss_cv:.4f}")
        self.save_results(fold_discriminator_losses, fold_generator_losses, avg_disc_loss_cv, avg_gen_loss_cv)

        # Save the final generator for visualization
        self.final_generator = generator


    def save_checkpoint(self, generator, discriminator, optimizer_G, optimizer_D, scheduler_G, scheduler_D, epoch, fold):
        """Saves a training checkpoint."""
        checkpoint_path = os.path.join(self.config['output_directory'], f'checkpoint_fold_{fold}_epoch_{epoch}.pth')
        torch.save({
            'epoch': epoch,
            'generator_state_dict': generator.state_dict(),
            'discriminator_state_dict': discriminator.state_dict(),
            'optimizer_G_state_dict': optimizer_G.state_dict(),
            'optimizer_D_state_dict': optimizer_D.state_dict(),
            'scheduler_G_state_dict': scheduler_G.state_dict(),
            'scheduler_D_state_dict': scheduler_D.state_dict(),
        }, checkpoint_path)
        print(f"Checkpoint saved for Fold {fold}, Epoch {epoch} to {checkpoint_path}")

    def load_checkpoint(self, generator, discriminator, optimizer_G, optimizer_D, scheduler_G, scheduler_D, fold):
        """Loads the latest checkpoint for a given fold."""
        checkpoint_files = glob.glob(os.path.join(self.config['output_directory'], f'checkpoint_fold_{fold}_*.pth'))
        if not checkpoint_files:
            print(f"No checkpoint found for Fold {fold}. Starting from scratch.")
            return 0

        # Find the latest checkpoint based on epoch number in the filename
        latest_checkpoint = max(checkpoint_files, key=lambda x: int(x.split('_')[-1].split('.')[0]))
        print(f"Loading checkpoint: {latest_checkpoint}")

        checkpoint = torch.load(latest_checkpoint, map_location=self.device)
        generator.load_state_dict(checkpoint['generator_state_dict'])
        discriminator.load_state_dict(checkpoint['discriminator_state_dict'])
        optimizer_G.load_state_dict(checkpoint['optimizer_G_state_dict'])
        optimizer_D.load_state_dict(checkpoint['optimizer_D_state_dict'])
        scheduler_G.load_state_dict(checkpoint['scheduler_G_state_dict'])
        scheduler_D.load_state_dict(checkpoint['scheduler_D_state_dict'])

        # Remove older checkpoints to save space
        for old_checkpoint in checkpoint_files:
            if old_checkpoint != latest_checkpoint:
                os.remove(old_checkpoint)

        return checkpoint['epoch']

    def train_gan(self, generator, discriminator, optimizer_G, optimizer_D, criterion, data_loader, scheduler_G, scheduler_D, scaler, fold, start_epoch):
        fold_disc_losses_epoch = []
        fold_gen_losses_epoch = []

        for epoch in range(start_epoch, self.config['num_epochs']):
            for i, real_sequences in enumerate(tqdm(data_loader, desc=f"Epoch {epoch+1}/{self.config['num_epochs']}")):
                real_sequences = real_sequences.to(self.device)
                batch_size = real_sequences.size(0)

                # Use label smoothing for real labels
                smooth_real_labels = torch.full((batch_size, 1), 1.0 - self.config['label_smoothing'], device=self.device)
                smooth_fake_labels = torch.full((batch_size, 1), self.config['label_smoothing'], device=self.device)

                # --- Train Discriminator ---
                # Use gradient accumulation
                with autocast():
                    # Train with real data
                    d_output_real = discriminator(real_sequences)
                    d_loss_real = criterion(d_output_real, smooth_real_labels)

                    # Train with fake data
                    fake_input_indices = torch.randint(0, self.config['vocab_size'], (batch_size, self.config['max_seq_len']), dtype=torch.long, device=self.device)
                    fake_sequences_logits = generator(fake_input_indices)
                    fake_sequences = torch.argmax(F.gumbel_softmax(fake_sequences_logits, tau=0.5, hard=True, dim=-1), dim=-1)

                    d_output_fake = discriminator(fake_sequences)
                    d_loss_fake = criterion(d_output_fake, smooth_fake_labels)

                    d_loss = d_loss_real + d_loss_fake

                # Scale the loss and backpropagate
                scaler.scale(d_loss).backward()

                if (i + 1) % self.config['gradient_accumulation_steps'] == 0:
                    scaler.unscale_(optimizer_D)
                    torch.nn.utils.clip_grad_norm_(discriminator.parameters(), self.config['gradient_clipping_value'])
                    scaler.step(optimizer_D)
                    scaler.update()
                    optimizer_D.zero_grad()

                # --- Train Generator ---
                if i % self.config['train_generator_every'] == 0:
                    with autocast():
                        gen_labels = torch.ones(batch_size, 1, device=self.device)

                        fake_input_indices_gen = torch.randint(0, self.config['vocab_size'], (batch_size, self.config['max_seq_len']), dtype=torch.long, device=self.device)
                        fake_sequences_logits_gen = generator(fake_input_indices_gen)
                        fake_sequences_gen = torch.argmax(F.gumbel_softmax(fake_sequences_logits_gen, tau=0.5, hard=True, dim=-1), dim=-1)

                        g_output = discriminator(fake_sequences_gen)
                        g_loss = criterion(g_output, gen_labels)

                    scaler.scale(g_loss).backward()

                    if (i + 1) % self.config['gradient_accumulation_steps'] == 0:
                        scaler.unscale_(optimizer_G)
                        torch.nn.utils.clip_grad_norm_(generator.parameters(), self.config['gradient_clipping_value'])
                        scaler.step(optimizer_G)
                        scaler.update()
                        optimizer_G.zero_grad()

            scheduler_G.step()
            scheduler_D.step()

            fold_disc_losses_epoch.append(d_loss.item())
            fold_gen_losses_epoch.append(g_loss.item())

            # Save checkpoint at the end of each epoch
            if (epoch + 1) % self.config['checkpoint_interval'] == 0:
                self.save_checkpoint(generator, discriminator, optimizer_G, optimizer_D, scheduler_G, scheduler_D, epoch + 1, fold)


        avg_fold_discriminator_loss = sum(fold_disc_losses_epoch) / len(fold_disc_losses_epoch)
        avg_fold_generator_loss = sum(fold_gen_losses_epoch) / len(fold_gen_losses_epoch)

        print(f"--- Finished Fold ---")
        print(f"Average Discriminator Loss: {avg_fold_discriminator_loss:.4f}")
        print(f"Average Generator Loss: {avg_fold_gen_loss:.4f}")

        return avg_fold_discriminator_loss, avg_fold_gen_loss

    def save_models(self, generator, discriminator, fold):
        """Saves the generator and discriminator models."""
        generator_path = os.path.join(self.config['output_directory'], f'generator_fold_{fold}.pth')
        discriminator_path = os.path.join(self.config['output_directory'], f'discriminator_fold_{fold}.pth')
        torch.save(generator.state_dict(), generator_path)
        torch.save(discriminator.state_dict(), discriminator_path)
        print(f"Saved models for fold {fold} to {generator_path} and {discriminator_path}")

    def save_results(self, fold_discriminator_losses, fold_generator_losses, avg_disc_loss_cv, avg_gen_loss_cv):
        """Saves the cross-validation results to a JSON file."""
        results = {
            'fold_discriminator_losses': fold_discriminator_losses,
            'fold_generator_losses': fold_generator_losses,
            'average_discriminator_loss_cv': avg_disc_loss_cv,
            'average_generator_loss_cv': avg_gen_loss_cv,
            'config': self.config
        }
        results_path = os.path.join(self.config['output_directory'], 'cross_validation_results.json')
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=4)
        print(f"Saved cross-validation results to {results_path}")

    def visualize_grid(self, grid_sequence, ax, title):
        """Visualizes a single flattened grid sequence."""
        # Find the separation token to split the input and output grids
        sep_token_index = (grid_sequence == 11).nonzero(as_tuple=True)[0]
        if sep_token_index.numel() > 0:
            sep_index = sep_token_index[0].item()
            # Split into input and output grids (ignoring the start token and sep token)
            output_flat = grid_sequence[sep_index + 1:]

            # Remove padding tokens
            output_flat = output_flat[output_flat != 0]

            # Try to reshape the output into a square grid for visualization
            size = int(np.sqrt(len(output_flat)))
            if size * size == len(output_flat):
                grid = output_flat.view(size, size).cpu().numpy()
                ax.imshow(grid, cmap='tab10', vmin=0, vmax=9)
                ax.set_title(title)
                ax.set_xticks(np.arange(size))
                ax.set_yticks(np.arange(size))
                ax.set_xticklabels([])
                ax.set_yticklabels([])
                ax.grid(which='both', color='gray', linestyle='-', linewidth=0.5)
            else:
                ax.set_title(f"{title}\n(Non-square grid)")
                ax.axis('off')
        else:
            ax.set_title(f"{title}\n(No separator token)")
            ax.axis('off')


    def generate_and_visualize_samples(self, num_samples=16):
        """
        Loads the final trained generator, generates new samples,
        and visualizes them.
        """
        print("\n--- Generating and Visualizing Samples ---")
        if self.final_generator is None:
            print("Generator not trained. Cannot generate samples.")
            return

        self.final_generator.eval()
        with torch.no_grad():
            # Generate a batch of random integer indices for the generator's input
            fake_input_indices = torch.randint(
                0, self.config['vocab_size'],
                (num_samples, self.config['max_seq_len']),
                dtype=torch.long, device=self.device
            )
            fake_sequences_logits = self.final_generator(fake_input_indices)

            # Apply Gumbel-Softmax and argmax to get discrete tokens
            fake_sequences_one_hot = F.gumbel_softmax(fake_sequences_logits, tau=0.5, hard=True, dim=-1)
            generated_sequences = torch.argmax(fake_sequences_one_hot, dim=-1)

        # Visualize the generated sequences
        fig, axes = plt.subplots(int(np.sqrt(num_samples)), int(np.sqrt(num_samples)), figsize=(12, 12))
        axes = axes.flatten()
        for i, ax in enumerate(axes):
            self.visualize_grid(generated_sequences[i], ax, f'Generated Grid {i+1}')

        plt.tight_layout()
        plt.show()


In [ ]:
# Assuming CONFIG, GANTrainer, etc., are already defined above this block.

# --- 6. MAIN EXECUTION ---
if __name__ == '__main__':
    # Initialize the GAN Trainer
    trainer = GANTrainer(CONFIG)

    # --- Data Loading and K-Fold Setup ---
    data_dir = os.path.join(CONFIG['base_data_path'], CONFIG['input_directory'])
    all_files = glob.glob(os.path.join(data_dir, '*.json'))

    if not all_files:
        print(f"Error: No JSON files found in directory: {data_dir}. Please check your path.")
        exit(1) # Exit if no data is found

    print(f"Found {len(all_files)} data files. Proceeding with K-Fold cross-validation.")
    kf = KFold(n_splits=CONFIG['n_splits'], shuffle=True, random_state=42)

    fold_discriminator_losses = []
    fold_generator_losses = []

    # --- K-Fold Cross-Validation with Robust Error Handling ---
    print("\n--- Starting K-Fold Cross-Validation ---")
    for fold, (train_index, val_index) in enumerate(kf.split(all_files)):
        print(f"\n--- Starting Fold {fold + 1}/{CONFIG['n_splits']} ---")
        train_file_paths = [all_files[i] for i in train_index]

        try:
            # Re-initialize dataset and loader for each fold (fresh start)
            train_dataset = ARCDataset(train_file_paths, CONFIG['max_seq_len'])
            train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=os.cpu_count() // 2 if os.cpu_count() else 0) # Use half CPU cores for data loading

            # Run training for the current fold
            avg_d_loss, avg_g_loss = trainer.train_one_fold(fold, train_loader)

            fold_discriminator_losses.append(avg_d_loss)
            fold_generator_losses.append(avg_g_loss)

            print(f"--- Finished Fold {fold + 1} successfully. ---")

        except torch.cuda.OutOfMemoryError as e:
            print(f"\n[ERROR] CUDA Out of Memory Error in Fold {fold + 1}: {e}")
            print("This usually means the batch size or sequence length is too large for your GPU.")
            print("Consider reducing CONFIG['batch_size'], CONFIG['max_seq_len'], CONFIG['d_model'], or CONFIG['num_layers'].")
            # Optionally, save partial results or exit gracefully
            # You might want to break here if OOM is unrecoverable for the current setup
            break
        except KeyboardInterrupt:
            print(f"\n[WARNING] Training interrupted by user in Fold {fold + 1}. Exiting gracefully.")
            # Save any intermediate results if needed before exiting
            break # Exit the K-Fold loop
        except Exception as e:
            print(f"\n[ERROR] An unexpected error occurred in Fold {fold + 1}: {e}")
            import traceback
            traceback.print_exc() # Print full traceback for debugging
            # Decide whether to continue to the next fold or break
            break # For critical errors, breaking might be safer

    print("\n--- Cross-Validation Summary ---")
    if fold_discriminator_losses: # Check if any folds completed successfully
        average_discriminator_loss_cv = sum(fold_discriminator_losses) / len(fold_discriminator_losses)
        average_generator_loss_cv = sum(fold_generator_losses) / len(fold_generator_losses)

        print(f"Average Discriminator Loss across {len(fold_discriminator_losses)} completed folds: {average_discriminator_loss_cv:.4f}")
        print(f"Average Generator Loss across {len(fold_generator_losses)} completed folds: {average_generator_loss_cv:.4f}")
    else:
        print("No folds completed successfully due to errors or interruption.")

    # --- Final Actions (e.g., Generate Samples) ---
    # This part would typically run *after* a successful training (or loading a pre-trained model).
    # Since K-Fold training can result in multiple models (one per fold, or the last one),
    # you might want to save and load a specific model here.
    # For simplicity, if we break due to error, we might skip this.

    print("\n--- Attempting to generate and visualize samples from the last trained model (if available) ---")
    try:
        # NOTE: For proper sample generation, you would need to save the Generator
        # from a completed fold (e.g., the last one or the best one) and load it here.
        # The current trainer.generate_and_visualize_samples() would need to
        # know which model to use. For now, it will use the Generator from the last
        # `trainer.train_one_fold` call, which might not be ideal if an error occurred.
        # Consider passing the final `generator` object from train_one_fold back,
        # or implement model saving/loading within the GANTrainer.

        # Placeholder: If you want to visualize from the 'best' model, you need to
        # modify train_one_fold to return the generator and discriminator, or save them.
        # For now, let's assume `trainer.generate_and_visualize_samples()` can
        # either load a default model or uses the last one configured.
        # If generate_and_visualize_samples is implemented to use the last model
        # trained in train_one_fold, it *might* work, but saving and loading is safer.
        # Example of how you'd explicitly save/load (requires changes to train_one_fold to return models)
        # torch.save(trainer.generator.state_dict(), f'generator_fold_{fold}.pth')
        # ... and then load here:
        # generator_for_vis = Generator(CONFIG).to(trainer.device)
        # generator_for_vis.load_state_dict(torch.load('generator_fold_X.pth'))
        # trainer.generate_and_visualize_samples(generator_for_vis)

        # Assuming generate_and_visualize_samples works with the last active generator
        trainer.generate_and_visualize_samples()
        print("Sample generation and visualization completed.")
    except Exception as e:
        print(f"[ERROR] Could not generate/visualize samples: {e}")
        import traceback
        traceback.print_exc()

    print("\n--- Main execution finished. ---")


--- Pre-loading and Caching All Data ---


Loading and Caching Data:   1%|          | 192/16144 [05:11<90:09:23, 20.35s/it]

In [ ]:
#visualize

In [ ]:
class RLModelSaverAndVisualizer:
    """
    A utility class to handle saving trained RL models and visualizing results.
    """
    def __init__(self, base_path):
        self.base_path = Path(base_path)

    def save_model(self, model, file_name, sub_dir='models'):
        """
        Saves a PyTorch model's state dictionary to the specified path.
        """
        save_dir = self.base_path / sub_dir
        save_dir.mkdir(parents=True, exist_ok=True)
        save_path = save_dir / file_name
        torch.save(model.state_dict(), save_path)
        print(f"Model saved to {save_path}")

    def save_plot(self, fig, file_name, sub_dir='plots'):
        """
        Saves a Matplotlib figure to the specified path.
        """
        save_dir = self.base_path / sub_dir
        save_dir.mkdir(parents=True, exist_ok=True)
        save_path = save_dir / file_name
        fig.savefig(save_path)
        print(f"Plot saved to {save_path}")

    def plot_results(self, all_d_losses, all_g_rewards):
        """
        Generates and saves plots of the training metrics.
        """
        # Plotting Discriminator Loss
        plt.figure(figsize=(10, 6))
        for i, fold_losses in enumerate(all_d_losses):
            plt.plot(fold_losses, label=f'Fold {i+1}')
        plt.title('Discriminator Loss over Epochs (Per Fold)')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)
        self.save_plot(plt.gcf(), 'discriminator_loss.png')
        plt.close()

        # Plotting Generator Reward
        plt.figure(figsize=(10, 6))
        for i, fold_rewards in enumerate(all_g_rewards):
            plt.plot(fold_rewards, label=f'Fold {i+1}')
        plt.title('Generator Reward over Epochs (Per Fold)')
        plt.xlabel('Epoch')
        plt.ylabel('Reward')
        plt.legend()
        plt.grid(True)
        self.save_plot(plt.gcf(), 'generator_reward.png')
        plt.close()

    def visualize_grid(self, grid_sequence, ax, title):
        """Visualizes a single flattened grid sequence."""
        # Find the separation token to split the input and output grids
        sep_token_index = (grid_sequence == 11).nonzero(as_tuple=True)[0]
        if sep_token_index.numel() > 0:
            sep_index = sep_token_index[0].item()
            # Split into input and output grids (ignoring the start token and sep token)
            output_flat = grid_sequence[sep_index + 1:]

            # Remove padding tokens
            output_flat = output_flat[output_flat != 0]

            # Try to reshape the output into a square grid for visualization
            size = int(np.sqrt(len(output_flat)))
            if size * size == len(output_flat):
                grid = output_flat.view(size, size).cpu().numpy()
                ax.imshow(grid, cmap='tab10', vmin=0, vmax=9)
                ax.set_title(title)
                ax.set_xticks(np.arange(size))
                ax.set_yticks(np.arange(size))
                ax.set_xticklabels([])
                ax.set_yticklabels([])
                ax.grid(which='both', color='gray', linestyle='-', linewidth=0.5)
            else:
                ax.set_title(f"{title}\n(Non-square grid)")
                ax.axis('off')
        else:
            ax.set_title(f"{title}\n(No separator token)")
            ax.axis('off')

    def generate_and_visualize_samples(self, generator, config, num_samples=16):
        """
        Generates new samples and visualizes them.
        """
        print("\n--- Generating and Visualizing Samples ---")
        generator.eval()
        device = next(generator.parameters()).device # Get the device from the model
        with torch.no_grad():
            # Generate a batch of random integer indices for the generator's input
            fake_input_indices = torch.randint(
                0, config['vocab_size'],
                (num_samples, config['max_seq_len']),
                dtype=torch.long, device=device
            )
            fake_sequences_logits = generator(fake_input_indices)

            # Apply Gumbel-Softmax and argmax to get discrete tokens
            fake_sequences_one_hot = F.gumbel_softmax(fake_sequences_logits, tau=0.5, hard=True, dim=-1)
            generated_sequences = torch.argmax(fake_sequences_one_hot, dim=-1)

        # Visualize the generated sequences
        fig, axes = plt.subplots(int(np.sqrt(num_samples)), int(np.sqrt(num_samples)), figsize=(12, 12))
        axes = axes.flatten()
        for i, ax in enumerate(axes):
            self.visualize_grid(generated_sequences[i], ax, f'Generated Grid {i+1}')

        plt.tight_layout()
        plt.show()